In [1]:
# import pandas as pd

# pd.set_option("display.max_rows", None)          # show all rows
# pd.set_option("display.max_columns", None)       # show all columns
# pd.set_option("display.max_colwidth", None)      # show full text in each cell
# pd.set_option("display.width", None)             # use full available width
# pd.set_option("display.expand_frame_repr", False)  # avoid wrapping into multiple blocks

In [2]:
import pandas as pd
import importlib
import utils   
importlib.reload(utils)
from utils import *
import ast
import pandas as pd
import pandas as pd

# pd.set_option("display.max_rows", None)          # show all rows
# pd.set_option("display.max_columns", None)       # show all columns
# pd.set_option("display.max_colwidth", None)      # show full text in each cell
# pd.set_option("display.width", None)             # use full available width
# pd.set_option("display.expand_frame_repr", False)  # avoid wrapping into multiple blocks

/Users/forough/VScodeProjects/VScode_serving/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tool_name = 'vllm'  #"repo_full_name"
# tool_name = 'sglang'  #"repo_full_name"
# tool_name = 'openllm'
# tool_name = 'tensorrt_llm'  #"repo_full_name"
# tool_name = 'lmdeploy'  #"repo_full_name"
# tool_name = 'routellm'
# tool_name = 'flashinfer'  #"repo_full_name"
# tool_name = 'lightllm'
# tool_name = 'fastdeploy'
# tool_name = 'easylm'
# tool_name = 'ramalama'
# tool_name = 'indexify'

file_path = f'/Users/forough/VScodeProjects/Vscode_serving/dataset/code_content_processed_results/{tool_name}_all_info_cleaned.csv'
df = pd.read_csv(file_path).sort_values(by="repo_full_name").reset_index(drop=True)
# df = pd.read_csv(file_path).sort_values(by="repoFull_name").reset_index(drop=True)
# sample_size = 100
# sample_size = len(df)
print(f"Total rows in {tool_name}: ", len(df))
# df = df.sample(n=sample_size, random_state=42)

print(f"Total rows in {tool_name}: ", len(df))
print(df['repo_full_name'].nunique())
# df

Total rows in vllm:  4750
Total rows in vllm:  4750
1821


# parameters

In [4]:
reference_params = get_tools_parameters("vllm")
df_with_refs = add_reference_columns(
    df_all=df,
    reference_params=reference_params,
    params_col="params",
    store="matched_substring"   # or "matched_item"
)
df_with_refs

,repo_full_name,Speculative decoding,Early existing,Cascade inference,Paged Attention,Continuous batching,Quantization,Chunked prefill,parallelism,Prefix caching,Multi-LoRA
0,0russwest0/Agent-R1,,,,,[max_num_batched_tokens],,[enable_chunked_prefill],[tensor_parallel_size],[enable_prefix_caching],
1,0russwest0/Agent-R1,,,,,,,,,,
2,0xLDF/Seg2Any,,,,,,,,,,
3,0xLDF/Seg2Any,,,,,,,,[tensor_parallel_size],,
4,0xk1h0/sven_modified,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...
4745,zq0x/zzvllm50,,,,,,,[enable_chunked_prefill],[tensor_parallel_size],[enable_prefix_caching],
4746,zq0x/zzvllm51,,,,,,,[enable_chunked_prefill],[tensor_parallel_size],[enable_prefix_caching],
4747,zulfikar2701/BRP-RAG-Prototype,,,,,,,,,,
4748,zwhe99/DeepMath,,,,,,,,[tensor_parallel_size],,


In [5]:
ref_cols = list(reference_params.keys())

df_with_refs_non_empty = df_with_refs[
    df_with_refs[ref_cols].apply(
        lambda row: any(isinstance(x, list) and len(x) > 0 for x in row),
        axis=1
    )
].copy()
# df_with_refs_non_empty.to_csv(f'{tool_name}_params_with_refs_non_empty.csv', index=False)
df_with_refs_non_empty

,repo_full_name,Speculative decoding,Early existing,Cascade inference,Paged Attention,Continuous batching,Quantization,Chunked prefill,parallelism,Prefix caching,Multi-LoRA
0,0russwest0/Agent-R1,,,,,[max_num_batched_tokens],,[enable_chunked_prefill],[tensor_parallel_size],[enable_prefix_caching],
3,0xLDF/Seg2Any,,,,,,,,[tensor_parallel_size],,
5,0xk1h0/sven_modified,,,,,,,,[tensor_parallel_size],,
7,18907305772/FuseAI,,,,,,,,[tensor_parallel_size],,
17,18907305772/FuseAI,,,,,,,,[tensor_parallel_size],,
...,...,...,...,...,...,...,...,...,...,...,...
4744,zq0x/zzvllm49,,,,,,,[enable_chunked_prefill],[tensor_parallel_size],[enable_prefix_caching],
4745,zq0x/zzvllm50,,,,,,,[enable_chunked_prefill],[tensor_parallel_size],[enable_prefix_caching],
4746,zq0x/zzvllm51,,,,,,,[enable_chunked_prefill],[tensor_parallel_size],[enable_prefix_caching],
4748,zwhe99/DeepMath,,,,,,,,[tensor_parallel_size],,


In [6]:
ref_cols = [c for c in df_with_refs.columns if c != "repo_full_name" and c in reference_params]

def is_non_empty_cell(x):
    return (
        (isinstance(x, list) and len(x) > 0) or
        (isinstance(x, str) and x.strip() != "")
    )

# Convert each reference column to True/False per row
mask_df = df_with_refs[["repo_full_name"] + ref_cols].copy()
for col in ref_cols:
    mask_df[col] = mask_df[col].apply(is_non_empty_cell)

# For each repo_full_name, count it once per column if any row is non-empty
unique_repo_counts = (
    mask_df.groupby("repo_full_name")[ref_cols]
    .any()
    .sum()
    .sort_values(ascending=False)
)

print(unique_repo_counts)

parallelism             923
Prefix caching          276
Continuous batching     163
Quantization            148
Chunked prefill         129
Multi-LoRA               90
Speculative decoding     11
Early existing            2
Cascade inference         0
Paged Attention           0
dtype: int64


In [7]:
import pandas as pd
import ast
import numpy as np

# df = df_with_refs_non_empty

value_cols = [col for col in df_with_refs_non_empty.columns if col != "repo_full_name"]


# Step 1: for each unique repo_full_name, mark whether each column is non-empty at least once
repo_level_presence = (
    df_with_refs_non_empty.groupby("repo_full_name")[value_cols]
      .agg(lambda s: any(is_nonempty_cell(x) for x in s))
)

# Step 2: build the combination of non-empty column names for each repo
repo_combinations = repo_level_presence.apply(
    lambda row: tuple(col for col in value_cols if row[col]),
    axis=1
)

# Step 3: keep only non-empty combinations and count occurrences
combination_counts = (
    repo_combinations[repo_combinations.apply(len) > 0]
    .value_counts()
    .reset_index()
)

combination_counts.columns = ["non_empty_column_combination", "unique_repo_count"]

# print(combination_counts)
combination_counts["non_empty_column_combination"] = combination_counts[
    "non_empty_column_combination"
].apply(lambda x: ", ".join(x))

combination_counts

,non_empty_column_combination,unique_repo_count
0,parallelism,567
1,Prefix caching,93
2,"parallelism, Prefix caching",69
3,"Quantization, parallelism",53
4,"Continuous batching, Chunked prefill, parallel...",40
5,Quantization,38
6,"parallelism, Multi-LoRA",33
7,"Continuous batching, parallelism",33
8,Multi-LoRA,26
9,"Continuous batching, Chunked prefill, parallelism",22


# imports

In [8]:
matches_df = (
    df["filtered_imports_features"]
    .apply(lambda d1: find_matches(d1, get_imports(tool_name), normalize=True))
    .apply(pd.Series)
)

# Add results to dataframe
df_out = df.copy()
df_out[matches_df.columns] = matches_df
# df_out

In [9]:
import pandas as pd
import ast
import numpy as np

match_cols = matches_df.columns.tolist()

merged_df = (
    df_out
    .groupby("repo_full_name")[match_cols]
    .agg(merge_lists)
    .reset_index()
)


merged_df.to_csv(f'/Users/forough/VScodeProjects/Vscode_serving/{tool_name}_merged_imports.csv', index=False)
# merged_df


In [10]:
import pandas as pd
import ast
import numpy as np

# df = merged_df

value_cols = [col for col in merged_df.columns if col != "repo_full_name"]


# Step 1: for each repo, reduce each column to True/False
repo_level_presence = (
    merged_df.groupby("repo_full_name")[value_cols]
      .agg(lambda s: any(is_nonempty_cell(x) for x in s))
)

# Step 2: count how many unique repos are non-empty in each column
nonempty_counts = repo_level_presence.sum().astype(int)

print(nonempty_counts)

CUDA                     22
Quantization             29
Multi-LoRA              196
Speculative decoding      3
Chunked prefill          23
parallelism             178
KVCaching                36
dtype: int64


In [11]:
# all columns except repo_full_name
value_cols = [col for col in merged_df.columns if col != "repo_full_name"]

# keep only rows where at least one non-empty list exists
merged_df_nonempty = merged_df[
    merged_df[value_cols].apply(
        lambda row: any(isinstance(x, list) and len(x) > 0 for x in row),
        axis=1
    )
].reset_index(drop=True)
# merged_df_nonempty.to_csv(f'{tool_name}_imports_merged_nonempty.csv', index=False)
merged_df_nonempty

,repo_full_name,CUDA,Quantization,Multi-LoRA,Speculative decoding,Chunked prefill,parallelism,KVCaching
0,0russwest0/Agent-R1,[],[],[],[],[],[parallel],[]
1,2noise/ChatTTS,[],[],[],[],[],[parallel],[]
2,46319943/DianpingAnalyzer,[],[],"[lora, lorarequest]",[],[],[],[]
3,A11en0/knowtrans,[],[],"[lora, lorarequest]",[],[],[],[]
4,AIDotNet/AntSK,[],[],"[lora, lorarequest]",[],[],[],[]
...,...,...,...,...,...,...,...,...
334,zhangshuibai/llada-reproduce,[],[],"[lora, lorarequest]",[],[],[],[]
335,zhengkid/Truth_Table_Logical_Reasoning,[],[],[],[],[],[parallel],[]
336,zitian-gao/one-shot-em,[],[],[],[],[],[parallel],[]
337,zjunlp/EasyInstruct,[],[],"[lora, lorarequest]",[],[],[],[]


In [12]:
import pandas as pd
import ast
import numpy as np

# df = merged_df_nonempty

value_cols = [col for col in merged_df_nonempty.columns if col != "repo_full_name"]



# Step 1: for each unique repo_full_name, mark whether each column is non-empty at least once
repo_level_presence = (
    merged_df_nonempty.groupby("repo_full_name")[value_cols]
      .agg(lambda s: any(is_nonempty_cell(x) for x in s))
)

# Step 2: build the combination of non-empty column names for each repo
repo_combinations = repo_level_presence.apply(
    lambda row: tuple(col for col in value_cols if row[col]),
    axis=1
)

# Step 3: keep only non-empty combinations and count occurrences
combination_counts = (
    repo_combinations[repo_combinations.apply(len) > 0]
    .value_counts()
    .reset_index()
)

combination_counts.columns = ["non_empty_column_combination", "unique_repo_count"]

# print(combination_counts)
combination_counts["non_empty_column_combination"] = combination_counts[
    "non_empty_column_combination"
].apply(lambda x: ", ".join(x))

combination_counts

,non_empty_column_combination,unique_repo_count
0,Multi-LoRA,137
1,parallelism,115
2,"Multi-LoRA, parallelism",27
3,Quantization,15
4,"CUDA, Multi-LoRA, Chunked prefill, parallelism...",10
5,"Multi-LoRA, Chunked prefill, parallelism, KVCa...",6
6,KVCaching,6
7,"CUDA, Quantization, Multi-LoRA, parallelism, K...",2
8,"CUDA, parallelism",2
9,"Quantization, parallelism",2


# merge results

In [13]:
import pandas as pd
import numpy as np
import ast

def merge_two_dataframes_by_repo(df1, df2, repo_col="repo_full_name"):
    def is_missing(x):
        if x is None:
            return True
        try:
            if pd.isna(x):
                return True
        except Exception:
            pass
        return False

    def to_list(x):
        """
        Convert a cell to a Python list.
        Supports:
        - actual lists/tuples/sets/ndarrays
        - stringified lists like "['a', 'b']"
        - single values -> [value]
        - empty/missing -> []
        """
        if is_missing(x):
            return []

        if isinstance(x, list):
            return x
        if isinstance(x, tuple):
            return list(x)
        if isinstance(x, set):
            return list(x)
        if isinstance(x, np.ndarray):
            return x.tolist()

        if isinstance(x, str):
            s = x.strip()
            if s == "" or s == "[]":
                return []
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return parsed
                if isinstance(parsed, tuple):
                    return list(parsed)
                if isinstance(parsed, set):
                    return list(parsed)
                if isinstance(parsed, np.ndarray):
                    return parsed.tolist()
                return [parsed]
            except Exception:
                return [s]

        return [x]

    def unique_preserve_order(seq):
        seen = set()
        out = []
        for item in seq:
            key = repr(item)
            if key not in seen:
                seen.add(key)
                out.append(item)
        return out

    def combine_cells(x1, x2):
        """
        Combine two cells as lists, remove duplicates, preserve order.
        """
        l1 = to_list(x1)
        l2 = to_list(x2)
        combined = unique_preserve_order(l1 + l2)
        return combined

    # --- column groups ---
    cols1 = [c for c in df1.columns if c != repo_col]
    cols2 = [c for c in df2.columns if c != repo_col]

    shared_cols = [c for c in cols1 if c in cols2]
    df1_only_cols = [c for c in cols1 if c not in cols2]
    df2_only_cols = [c for c in cols2 if c not in cols1]

    final_cols = [repo_col] + shared_cols + df1_only_cols + df2_only_cols

    # --- make repo_full_name unique inside each dataframe if needed ---
    # for repeated repo_full_name inside same dataframe, keep first non-missing value
    def collapse_single_df(df, repo_col):
        def first_non_missing(series):
            for v in series:
                if not is_missing(v):
                    return v
            return np.nan

        agg_dict = {col: first_non_missing for col in df.columns if col != repo_col}
        return df.groupby(repo_col, as_index=False).agg(agg_dict)

    df1_u = collapse_single_df(df1, repo_col)
    df2_u = collapse_single_df(df2, repo_col)

    # --- outer merge on repo_full_name ---
    merged = df1_u.merge(
        df2_u,
        on=repo_col,
        how="outer",
        suffixes=("_df1", "_df2")
    )

    # --- build final dataframe ---
    result = pd.DataFrame()
    result[repo_col] = merged[repo_col]

    # shared columns: combine lists from both dataframes
    for col in shared_cols:
        result[col] = merged.apply(
            lambda row: combine_cells(row.get(f"{col}_df1"), row.get(f"{col}_df2")),
            axis=1
        )

    # df1-only columns: copy from df1
    for col in df1_only_cols:
        result[col] = merged[f"{col}"]

    # df2-only columns: copy from df2
    for col in df2_only_cols:
        result[col] = merged[f"{col}"]

    # reorder columns
    result = result[final_cols].copy()
    result = result.fillna("")

    # reorder columns
    result = result[final_cols].copy()

    # replace NaN with ""
    result = result.fillna("")

    # replace empty lists [] with ""
    result = result.applymap(lambda x: "" if isinstance(x, (list, tuple, set, np.ndarray)) and len(x) == 0 else x)

    return result

In [14]:
final_df = merge_two_dataframes_by_repo(df_with_refs_non_empty, merged_df_nonempty)

final_df = final_df.rename(
    columns={col: f"{tool_name}_{col}" for col in final_df.columns if col != "repo_full_name"}
)

final_df = final_df.fillna("")

# add new column
final_df["tool_name"] = tool_name

# store dataframe using tool_name as key
# final_df[tool_name] = final_df



final_df.to_csv(f'/Users/forough/VScodeProjects/VScode_serving/dataset/merged_used_imports_params/{tool_name}_final_df.csv', index=False)
final_df

,repo_full_name,vllm_Speculative decoding,vllm_Quantization,vllm_Chunked prefill,vllm_parallelism,vllm_Multi-LoRA,vllm_Early existing,vllm_Cascade inference,vllm_Paged Attention,vllm_Continuous batching,vllm_Prefix caching,vllm_CUDA,vllm_KVCaching,tool_name
0,0russwest0/Agent-R1,,,[enable_chunked_prefill],"[tensor_parallel_size, parallel]",,,,,[max_num_batched_tokens],[enable_prefix_caching],,,vllm
1,0xLDF/Seg2Any,,,,[tensor_parallel_size],,,,,,,,,vllm
2,0xk1h0/sven_modified,,,,[tensor_parallel_size],,,,,,,,,vllm
3,18907305772/FuseAI,,,,[tensor_parallel_size],,,,,,,,,vllm
4,1Wata/mllmTracker,,,,,,,,,,[enable_prefix_caching],,,vllm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1221,zq0x/zzvllm49,,,[enable_chunked_prefill],[tensor_parallel_size],,,,,,[enable_prefix_caching],,,vllm
1222,zq0x/zzvllm50,,,[enable_chunked_prefill],[tensor_parallel_size],,,,,,[enable_prefix_caching],,,vllm
1223,zq0x/zzvllm51,,,[enable_chunked_prefill],[tensor_parallel_size],,,,,,[enable_prefix_caching],,,vllm
1224,zwhe99/DeepMath,,,,[tensor_parallel_size],,,,,,,,,vllm


In [15]:
# final_df.to_csv(f'{tool_name}_final_merged.csv', index=False)


In [16]:
import pandas as pd
import numpy as np
import ast

nonempty_counts = (
    final_df.drop(columns=["repo_full_name"])
          .apply(lambda col: col.apply(is_nonempty_cell).sum())
)

# print(nonempty_counts)
print(nonempty_counts.sort_values(ascending=False))

tool_name                    1226
vllm_parallelism              977
vllm_Prefix caching           236
vllm_Multi-LoRA               202
vllm_Quantization             140
vllm_Continuous batching      124
vllm_Chunked prefill          121
vllm_KVCaching                 36
vllm_CUDA                      22
vllm_Speculative decoding       9
vllm_Early existing             1
vllm_Cascade inference          0
vllm_Paged Attention            0
dtype: int64


In [17]:
# Total number of unique respos
print(df['repo_full_name'].nunique())


1821


In [18]:
import pandas as pd
import numpy as np
import ast
from itertools import combinations
from collections import Counter

# exclude repo_full_name
cols = [c for c in final_df.columns if c != "repo_full_name"]

# boolean mask
mask = final_df[cols].apply(lambda col: col.map(is_nonempty_cell))

# count combinations
counter = Counter()

for _, row in mask.iterrows():
    nonempty_cols = [col for col, val in row.items() if val]
    
    for r in range(1, len(nonempty_cols) + 1):
        for combo in combinations(nonempty_cols, r):
            counter[combo] += 1

# final DataFrame (ONLY required columns)
result_final_df = pd.DataFrame([
    {
        "column_combination": combo,
        "occurrence in unique repos": count
    }
    for combo, count in counter.items()
])

# sort by occurrence
result_final_df = result_final_df.sort_values(
    by="occurrence in unique repos",
    ascending=False
).reset_index(drop=True)
# pd.set_option("display.max_rows", None)          # show all rows
# pd.set_option("display.max_columns", None)       # show all columns
# pd.set_option("display.max_colwidth", None)      # show full text in each cell
# pd.set_option("display.width", None)             # use full available width
# pd.set_option("display.expand_frame_repr", False)  # avoid wrapping into multiple blocks

print("unique number of repos",df['repo_full_name'].nunique())

result_final_df

unique number of repos 1821


,column_combination,occurrence in unique repos
0,"(tool_name,)",1226
1,"(vllm_parallelism,)",977
2,"(vllm_parallelism, tool_name)",977
3,"(vllm_Prefix caching,)",236
4,"(vllm_Prefix caching, tool_name)",236
...,...,...
598,"(vllm_Quantization, vllm_Chunked prefill, vllm...",1
599,"(vllm_Quantization, vllm_Chunked prefill, vllm...",1
600,"(vllm_Quantization, vllm_Chunked prefill, vllm...",1
601,"(vllm_Quantization, vllm_Chunked prefill, vllm...",1
